In [1]:
%cd /content
!rm -rf mccain-internship
!git clone https://github.com/PushkargithubCSE/mccain-internship.git
%cd mccain-internship/week-3/potato-finetuning
!pwd

/content
Cloning into 'mccain-internship'...
remote: Enumerating objects: 542, done.
remote: Counting objects: 100% (542/542), done.
remote: Compressing objects: 100% (397/397), done.
remote: Total 542 (delta 135), reused 489 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (542/542), 23.47 MiB | 23.17 MiB/s, done.
Resolving deltas: 100% (135/135), done.
/content/mccain-internship/week-3/potato-finetuning
/content/mccain-internship/week-3/potato-finetuning


In [2]:
!pip install -q "rfdetr[train,metrics,loggers]"
!pip install -q roboflow python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 11.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.6/376.6 kB 8.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 9.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 19.9 MB/s

In [3]:
import os
from dotenv import load_dotenv

PROJECT_PATH = "/content/mccain-internship/week-3/potato-finetuning"
load_dotenv(f"{PROJECT_PATH}/.env")

api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("ROBOFLOW_API_KEY is missing from the project .env file")

✓ .env created


In [4]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv(f"{PROJECT_PATH}/.env")
api_key = os.getenv('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=api_key)
project = rf.workspace("pushkar-chandra").project("object-detection-yntmm")
version = project.version(1)
dataset = version.download("coco")

print(f"✓ Dataset ready at: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Object-Detection-1 in coco:: 100%|██████████| 21/21 [00:00<00:00, 4288.24it/s]

✓ Dataset ready at: /content/mccain-internship/week-3/potato-finetuning/Object-Detection-1


In [5]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU: Tesla T4


In [6]:
!pip install -q huggingface_hub

In [7]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(f"{PROJECT_PATH}/.env")
hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise RuntimeError("HUGGINGFACE_TOKEN is missing from the project .env file")
login(token=hf_token)

print("Logged into Hugging Face")

✓ Logged into Hugging Face


In [8]:
!pip install -q pyngrok

In [9]:
import os
from dotenv import load_dotenv
from pyngrok import ngrok
import subprocess
import time

PROJECT_PATH = "/content/mccain-internship/week-3/potato-finetuning"
OUTPUT_DIR = f"{PROJECT_PATH}/checkpoints/rfdetr_run1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

load_dotenv(f"{PROJECT_PATH}/.env")
ngrok_token = os.getenv("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("NGROK_AUTHTOKEN is missing from the project .env file")
ngrok.set_auth_token(ngrok_token)

!pkill -f tensorboard
ngrok.kill()

subprocess.Popen(['tensorboard', '--logdir', OUTPUT_DIR, '--host', '0.0.0.0', '--port', '6006'])
time.sleep(5)

✓ Open this in your browser NOW (will populate once training starts): NgrokTunnel: "https://erin-cataclysmic-tegularly.ngrok-free.dev" -> "http://localhost:6006"


In [10]:
from rfdetr import RFDETRBase
from dotenv import load_dotenv
from huggingface_hub import login, HfApi, create_repo
import os

PROJECT_PATH = "/content/mccain-internship/week-3/potato-finetuning"
OUTPUT_DIR = f"{PROJECT_PATH}/checkpoints/rfdetr_run1"

# --- TRAIN ---
model = RFDETRBase()
model.train(
    dataset_dir=dataset.location,
    epochs=30,
    batch_size=2,
    grad_accum_steps=8,
    lr=1e-4,
    output_dir=OUTPUT_DIR,
    early_stopping=True
)
print("Training complete.")

# --- UPLOAD IMMEDIATELY, SAME CELL ---
load_dotenv(f"{PROJECT_PATH}/.env")
hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise RuntimeError("HUGGINGFACE_TOKEN is missing from the project .env file")
login(token=hf_token)

REPO_ID = "pushkar0002/finetune"

api = HfApi()
create_repo(repo_id=REPO_ID, exist_ok=True, repo_type="model")
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, repo_type="model")

print(f"Checkpoint safely uploaded to: https://huggingface.co/{REPO_ID}")

[2026-08-27 09:44:23] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-base.pth


The `RFDETRBase` was deprecated since v1.7.0. It will be removed in v2.0.0.


/root/.roboflow/models/rf-detr-base.pth:   0%|          | 0.00/355M [00:00<?, ?iB/s]

[2026-08-27 09:44:29] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-base.pth
[2026-08-27 09:44:31] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-base.pth already exists with correct MD5 hash.
[2026-08-27 09:44:34] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-base.pth already exists with correct MD5 hash.


[2026-08-27 09:44:36] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 2. The detection head will be re-initialized to 2 classes.
INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Output()

INFO:pytorch_lightning.callbacks.early_stopping:Metric __rfdetr_effective_map__ improved. New best score: 0.413


[2026-08-27 09:45:03] [INFO] rf-detr - Best regular checkpoint saved to /content/mccain-internship/week-3/potato-finetuning/checkpoints/rfdetr_run1/checkpoint_best_regular.pth (epoch 0, monitor=val/mAP_50_95, value=0.413019)
[2026-08-27 09:45:05] [INFO] rf-detr - Best EMA mAP improved to 0.4018 (epoch 0)


INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric __rfdetr_effective_map__ did not improve in the last 10 records. Best score: 0.413. Signaling Trainer to stop.


[2026-08-27 09:50:39] [INFO] rf-detr - Best total checkpoint saved from regular (regular=0.4130, ema=0.4018)
✓ Training complete.
✓ Checkpoint safely uploaded to: https://huggingface.co/pushkar0002/finetune


In [11]:
import os

PROJECT_PATH = "/content/mccain-internship/week-3/potato-finetuning"

print("PROJECT_PATH exists:", os.path.exists(PROJECT_PATH))

checkpoints_dir = f"{PROJECT_PATH}/checkpoints"
print("checkpoints/ exists:", os.path.exists(checkpoints_dir))

if os.path.exists(checkpoints_dir):
    print("Contents of checkpoints/:", os.listdir(checkpoints_dir))
else:
    print("checkpoints/ folder doesn't exist at all")

PROJECT_PATH exists: True
checkpoints/ exists: True
Contents of checkpoints/: ['rfdetr_run1']


In [12]:
import os

PROJECT_PATH = "/content/mccain-internship/week-3/potato-finetuning"
OUTPUT_DIR = f"{PROJECT_PATH}/checkpoints/rfdetr_run1"

print("Exists:", os.path.exists(OUTPUT_DIR))
if os.path.exists(OUTPUT_DIR):
    print("Files:", os.listdir(OUTPUT_DIR))

Exists: True
Files: ['metrics.csv', 'events.out.tfevents.1787823878.414c3b9555a3.3433.0', 'checkpoint_9.ckpt', 'checkpoint_best_ema.pth', 'hparams.yaml', 'last_ema.pth', 'last.ckpt', 'checkpoint_best_total.pth', 'checkpoint_best_regular.pth', 'training_config.json']


In [ ]:
!du -sh {OUTPUT_DIR}